<a href="https://colab.research.google.com/github/pranitgarje/Neural_Networks/blob/main/makemore_part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn.functional as F


In [5]:
words=open("names.txt","r").read().splitlines()

In [9]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [18]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
vocab_size=len(itos)

In [19]:
vocab_size

27

In [29]:
block_size=3
def build_dataset(words):
  X,Y=[],[]
  for w in words :
    context=[0]*block_size
    for ch in w+'.':
      ix=stoi[ch]
      X.append(context)
      Y.append(ix)
      context=context[1:]+[ix]
  X=torch.tensor(X)
  Y=torch.tensor(Y)
  return X,Y
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

In [30]:
g = torch.Generator().manual_seed(2147483647)
n_embd=10
n_hidden=200
C = torch.randn((vocab_size, n_embd), generator=g)
W1=torch.randn((n_embd*block_size,n_hidden),generator=g)*(5/3)/((n_embd*block_size)**0.5)
# b1=torch.randn(n_hidden,generator=g)
W2=torch.randn((n_hidden,vocab_size),generator=g)
b2=torch.randn(vocab_size,generator=g)

bngain=torch.ones((1,n_hidden))
bnbias=torch.zeros((1,n_hidden))
bnmean_running=torch.zeros((1,n_hidden))
bnstd_running=torch.ones((1,n_hidden))

parameters=[C,W1,W2,b2,bngain,bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad=True

12097


In [35]:
max_steps=200000
batch_size=32
lossi=[]

for i in range(max_steps):
  ix=torch.randint(0,Xtr.shape[0],(batch_size,))
  Xb,Yb=Xtr[ix],Ytr[ix]
  emb=C[Xb]
  embcat=emb.view(emb.shape[0],-1)

  hpreact=embcat@W1

  bnmeani=hpreact.mean(0,keepdim=True)
  bnstdi=hpreact.std(0,keepdim=True)
  hpreact=bngain*(hpreact-bnmeani)/bnstdi+bnbias

  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

  h=torch.tanh(hpreact)
  logits=h@W2+b2
  loss=F.cross_entropy(logits,Yb)

  for p in parameters :
    p.grad=None

  loss.backward()
   # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())





      0/ 200000: 18.9938
  10000/ 200000: 2.4293
  20000/ 200000: 2.1832
  30000/ 200000: 2.5276
  40000/ 200000: 2.5574
  50000/ 200000: 2.4317
  60000/ 200000: 2.2121
  70000/ 200000: 2.2703
  80000/ 200000: 2.4074
  90000/ 200000: 1.9716
 100000/ 200000: 2.1081
 110000/ 200000: 2.1772
 120000/ 200000: 2.1904
 130000/ 200000: 1.7334
 140000/ 200000: 2.2231
 150000/ 200000: 2.1797
 160000/ 200000: 2.1346
 170000/ 200000: 2.2656
 180000/ 200000: 1.9233
 190000/ 200000: 1.9780
